# CV-STQA — M3: Patch CNN + Average Pooling

**Goal:** implement the manual's M3 ablation: local spatial patches → CNN → local quality → average pooling.

- 16 uniformly sampled frames/video
- 7×7 = 49 patches/frame
- 64×64 local crops
- ResNet-50 patch CNN
- Average spatial pooling
- Average temporal pooling
- Six quality dimensions: TSV, B, SR, S, U, O

This notebook is designed to be memory-safe on Kaggle. It processes one video at a time and patches in small GPU batches, while decoding each video only once.


In [1]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

import os, random, json, time, gc
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr, pearsonr, kendalltau
from sklearn.metrics import mean_squared_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_FRAMES = 16
GRID_SIZE = 7
NUM_PATCHES = GRID_SIZE * GRID_SIZE
PATCH_SIZE = 64

# Keep this small for Kaggle memory safety.
PATCH_BATCH_SIZE = 32

# First M3 stage: frozen pretrained ResNet-50 + trainable quality head.
FREEZE_BACKBONE = True

EPOCHS = 2
LEARNING_RATE = 1e-3

TARGETS = ["TSV", "B", "SR", "S", "U", "O"]

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


## 2. Locate the two Kaggle inputs

The code searches `/kaggle/input` instead of relying on a fragile `VIDEO_DIR` variable.

In [2]:
# ============================================================
# 2. LOCATE DATA
# ============================================================

MOS_FILE = None
VIDEO_FILES = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        full = os.path.join(root, file)

        if file == "MOS.txt":
            MOS_FILE = full

        if file.lower().endswith(".mp4"):
            VIDEO_FILES.append(full)

if MOS_FILE is None:
    raise FileNotFoundError("MOS.txt was not found under /kaggle/input.")

print("MOS_FILE:", MOS_FILE)
print("Number of .mp4 files:", len(VIDEO_FILES))

if not VIDEO_FILES:
    raise FileNotFoundError("No .mp4 files were found under /kaggle/input.")


MOS_FILE: /kaggle/input/datasets/miruthulajmm/vqa-mos/MOS.txt
Number of .mp4 files: 1000


## 3. Parse the actual MOS format

Your file is:

`video.mp4,TSV,B,SR,S,U,O`

For example:

`1-14.mp4,0.9235,1.3311,1.356,1.0388,0.92656,1.846`

In [3]:
# ============================================================
# 3. PARSE MOS.TXT + MATCH VIDEOS
# ============================================================

video_map = {
    Path(p).stem: p
    for p in VIDEO_FILES
}

rows = []

with open(MOS_FILE, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        parts = line.split(",")

        if len(parts) != 7:
            continue

        filename = parts[0].strip()
        video_id = Path(filename).stem

        try:
            scores = [float(x) for x in parts[1:7]]
        except ValueError:
            continue

        if video_id not in video_map:
            continue

        rows.append({
            "video_id": video_id,
            "video_path": video_map[video_id],
            "TSV": scores[0],
            "B": scores[1],
            "SR": scores[2],
            "S": scores[3],
            "U": scores[4],
            "O": scores[5],
        })

df = pd.DataFrame(rows).drop_duplicates("video_id").reset_index(drop=True)

print("Matched videos:", len(df))
display(df.head())

if len(df) == 0:
    raise RuntimeError("No MOS entries matched the video files.")


Matched videos: 1000


,video_id,video_path,TSV,B,SR,S,U,O
0,1-14,/kaggle/input/datasets/miruthulajmm/mac-vqa-vi...,0.92350,1.3311,1.3560,1.03880,0.92656,1.8460
1,1-18,/kaggle/input/datasets/miruthulajmm/mac-vqa-vi...,0.94540,1.5686,0.8950,0.74001,1.36560,2.4560
2,1-22,/kaggle/input/datasets/miruthulajmm/mac-vqa-vi...,0.60707,1.7066,1.6000,0.00000,0.25600,1.4056
3,1-4,/kaggle/input/datasets/miruthulajmm/mac-vqa-vi...,2.56440,1.8566,2.4500,1.96870,2.90560,4.1097
4,10-12,/kaggle/input/datasets/miruthulajmm/mac-vqa-vi...,1.65490,1.7560,1.4035,1.87150,1.50000,2.8304


## 4. Fixed 80/20 video-level split

The split is performed before any frame/patch extraction, preventing frame/patch leakage between train and test.

In [4]:
# ============================================================
# 4. TRAIN / TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv("/kaggle/working/M3_train_split.csv", index=False)
test_df.to_csv("/kaggle/working/M3_test_split.csv", index=False)

print("Total:", len(df))
print("Train:", len(train_df))
print("Test :", len(test_df))


Total: 1000
Train: 800
Test : 200


## 5. Efficient video decoding and 7×7 patch extraction

Each video is opened **once**. We sample 16 frames and create 49 patches per frame. Patches are kept at 64×64 until small GPU batches are formed.

In [5]:
# ============================================================
# 5. FRAME + PATCH EXTRACTION
# ============================================================

def sample_frames_and_patches(video_path, num_frames=NUM_FRAMES,
                              grid_size=GRID_SIZE, patch_size=PATCH_SIZE):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total <= 0:
        cap.release()
        raise RuntimeError(f"Invalid frame count: {video_path}")

    indices = np.linspace(0, total - 1, num_frames).astype(int)

    all_patches = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()

        if not ok or frame is None:
            continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        h, w = frame.shape[:2]

        # 7x7 grid with evenly spaced centers.
        ys = np.linspace(0, max(0, h - 1), grid_size + 2)[1:-1]
        xs = np.linspace(0, max(0, w - 1), grid_size + 2)[1:-1]

        half = patch_size // 2

        for cy in ys.astype(int):
            for cx in xs.astype(int):
                y0 = max(0, cy - half)
                x0 = max(0, cx - half)

                y1 = y0 + patch_size
                x1 = x0 + patch_size

                if y1 > h:
                    y1 = h
                    y0 = max(0, h - patch_size)

                if x1 > w:
                    x1 = w
                    x0 = max(0, w - patch_size)

                patch = frame[y0:y1, x0:x1]

                if patch.shape[0] != patch_size or patch.shape[1] != patch_size:
                    patch = cv2.resize(
                        patch,
                        (patch_size, patch_size),
                        interpolation=cv2.INTER_LINEAR
                    )

                all_patches.append(patch)

    cap.release()

    # Shape: [frames, 49, H, W, 3]
    expected = num_frames * grid_size * grid_size

    if len(all_patches) != expected:
        raise RuntimeError(
            f"Expected {expected} patches but got {len(all_patches)} "
            f"for {video_path}"
        )

    arr = np.stack(all_patches).reshape(
        num_frames, grid_size * grid_size,
        patch_size, patch_size, 3
    )

    return arr


## 6. ResNet-50 patch CNN

The CNN extracts local patch representations. A small trainable head predicts the six quality dimensions for each patch. Spatial quality is then averaged across the 49 patches.

In [6]:
# ============================================================
# 6. RESNET-50 PATCH CNN
# ============================================================

weights = models.ResNet50_Weights.DEFAULT
backbone = models.resnet50(weights=weights)

# Remove ImageNet classifier.
feature_dim = backbone.fc.in_features
backbone.fc = nn.Identity()

if FREEZE_BACKBONE:
    for p in backbone.parameters():
        p.requires_grad = False

class PatchQualityCNN(nn.Module):
    def __init__(self, backbone, feature_dim, num_outputs=6):
        super().__init__()
        self.backbone = backbone
        self.quality_head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_outputs)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.quality_head(features)

model = PatchQualityCNN(
    backbone=backbone,
    feature_dim=feature_dim,
    num_outputs=len(TARGETS)
).to(DEVICE)

print(model.__class__.__name__)
print("Feature dimension:", feature_dim)
print("Backbone frozen:", FREEZE_BACKBONE)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 241MB/s]


PatchQualityCNN
Feature dimension: 2048
Backbone frozen: True


In [7]:
# ============================================================
# 7. PREPROCESSING
# ============================================================

# ResNet-50 receives 224x224 inputs.
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def preprocess_patch_batch(patches):
    tensors = [preprocess(p) for p in patches]
    return torch.stack(tensors)


## 8. M3 video forward pass

This is the important efficiency fix.

We **do not** create a giant tensor containing all 784 patches at once. Patches are sent through ResNet-50 in chunks of 32. The resulting local quality predictions are averaged over the 49 spatial patches and then over the 16 sampled frames.

In [8]:
# ============================================================
# 8. MEMORY-SAFE M3 FORWARD PASS
# ============================================================

@torch.no_grad()
def predict_video(model, video_path):
    model.eval()

    patches = sample_frames_and_patches(video_path)

    # [16, 49, 64, 64, 3]
    frame_quality = []

    for frame_idx in range(NUM_FRAMES):

        frame_patches = patches[frame_idx]

        patch_outputs = []

        for start in range(0, NUM_PATCHES, PATCH_BATCH_SIZE):

            batch_np = frame_patches[
                start:start + PATCH_BATCH_SIZE
            ]

            batch = preprocess_patch_batch(batch_np).to(
                DEVICE,
                non_blocking=True
            )

            out = model(batch)

            patch_outputs.append(out.detach().cpu())

            del batch, out

        patch_outputs = torch.cat(patch_outputs, dim=0)

        # Average pooling across 49 spatial patches.
        frame_score = patch_outputs.mean(dim=0)

        frame_quality.append(frame_score)

    # Average pooling across sampled frames.
    video_score = torch.stack(frame_quality).mean(dim=0)

    return video_score.numpy()


In [9]:
# ============================================================
# 9. QUICK SANITY TEST — ONE VIDEO
# ============================================================

test_path = train_df.iloc[0]["video_path"]

start_time = time.time()

sample_pred = predict_video(model, test_path)

print("Video:", train_df.iloc[0]["video_id"])
print("Prediction shape:", sample_pred.shape)
print("Prediction:", sample_pred)
print("Time:", round(time.time() - start_time, 2), "seconds")


Video: 2-19
Prediction shape: (6,)
Prediction: [ 0.09424841 -0.01952185 -0.11361744  0.01244517  0.06206191 -0.10773233]
Time: 6.34 seconds


## 10. Training strategy

Because the manual recommends transfer learning with most of the backbone initially frozen, this first M3 stage trains the quality head while keeping ResNet-50 frozen.

In [10]:
# ============================================================
# 10. TRAINING SETUP
# ============================================================

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

criterion = nn.MSELoss()

print(
    "Trainable parameters:",
    sum(p.numel() for p in trainable_params)
)


Trainable parameters: 526086


## 11. Efficient training

Each video produces one six-dimensional prediction after patch/frame average pooling. The loss is computed against the six MOS targets.

In [11]:
# ============================================================
# 11. TRAIN
# ============================================================

history = []

for epoch in range(EPOCHS):

    model.train()

    # Backbone remains frozen; only quality head is optimized.
    running_loss = 0.0

    progress = tqdm(
        train_df.itertuples(index=False),
        total=len(train_df),
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for row in progress:

        optimizer.zero_grad(set_to_none=True)

        # Need gradients through quality head.
        patches = sample_frames_and_patches(row.video_path)

        frame_outputs = []

        for frame_idx in range(NUM_FRAMES):

            frame_patches = patches[frame_idx]
            patch_outputs = []

            for start in range(0, NUM_PATCHES, PATCH_BATCH_SIZE):

                batch_np = frame_patches[
                    start:start + PATCH_BATCH_SIZE
                ]

                batch = preprocess_patch_batch(batch_np).to(DEVICE)

                out = model(batch)

                patch_outputs.append(out)

            patch_outputs = torch.cat(patch_outputs, dim=0)

            # M3 spatial average pooling.
            frame_score = patch_outputs.mean(dim=0)

            frame_outputs.append(frame_score)

        # M3 temporal averaging.
        video_pred = torch.stack(frame_outputs).mean(dim=0)

        target = torch.tensor(
            [getattr(row, c) for c in TARGETS],
            dtype=torch.float32,
            device=DEVICE
        )

        loss = criterion(video_pred, target)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        progress.set_postfix(loss=f"{loss.item():.4f}")

        del patches, frame_outputs, video_pred, target, loss

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    epoch_loss = running_loss / len(train_df)

    history.append({
        "epoch": epoch + 1,
        "train_loss": epoch_loss
    })

    print(f"Epoch {epoch + 1}: train loss = {epoch_loss:.6f}")


Epoch 1/2:   0%|          | 0/800 [00:00<?, ?it/s]

Epoch 1: train loss = 0.728947


Epoch 2/2:   0%|          | 0/800 [00:00<?, ?it/s]

Epoch 2: train loss = 0.641065


## 12. Evaluation metrics

Report SRCC, PLCC, KRCC and RMSE for all six dimensions, matching the manual's evaluation requirement.

In [12]:
# ============================================================
# 12. EVALUATION
# ============================================================

def evaluate_model(model, data):

    model.eval()

    predictions = []
    ground_truth = []
    ids = []

    with torch.no_grad():

        for row in tqdm(
            data.itertuples(index=False),
            total=len(data),
            desc="Evaluating"
        ):

            pred = predict_video(model, row.video_path)

            gt = np.array(
                [getattr(row, c) for c in TARGETS],
                dtype=np.float32
            )

            predictions.append(pred)
            ground_truth.append(gt)
            ids.append(row.video_id)

    predictions = np.asarray(predictions)
    ground_truth = np.asarray(ground_truth)

    metrics = []

    for i, target in enumerate(TARGETS):

        y_true = ground_truth[:, i]
        y_pred = predictions[:, i]

        srcc = spearmanr(y_true, y_pred).statistic
        plcc = pearsonr(y_true, y_pred).statistic
        krcc = kendalltau(y_true, y_pred).statistic
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))

        metrics.append({
            "dimension": target,
            "SRCC": srcc,
            "PLCC": plcc,
            "KRCC": krcc,
            "RMSE": rmse
        })

    metrics_df = pd.DataFrame(metrics)

    pred_df = pd.DataFrame(
        predictions,
        columns=[f"pred_{x}" for x in TARGETS]
    )

    gt_df = pd.DataFrame(
        ground_truth,
        columns=[f"true_{x}" for x in TARGETS]
    )

    result_df = pd.concat(
        [
            pd.DataFrame({"video_id": ids}),
            gt_df,
            pred_df
        ],
        axis=1
    )

    return metrics_df, result_df


In [13]:
# ============================================================
# 13. TEST SET EVALUATION
# ============================================================

test_metrics, test_predictions = evaluate_model(
    model,
    test_df
)

print("TEST METRICS")
display(test_metrics)

test_metrics.to_csv(
    "/kaggle/working/M3_test_metrics.csv",
    index=False
)

test_predictions.to_csv(
    "/kaggle/working/M3_test_predictions.csv",
    index=False
)


Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

TEST METRICS


,dimension,SRCC,PLCC,KRCC,RMSE
0,TSV,0.602592,0.604299,0.430339,0.892249
1,B,0.592797,0.588056,0.421560,0.965277
2,SR,0.342520,0.284730,0.229696,0.901791
3,S,0.614580,0.614717,0.442172,0.869813
4,U,0.618182,0.602866,0.438269,0.927737
5,O,0.637269,0.623524,0.459868,1.500961


## 14. Save M3 artifacts

These are the files to keep for comparison with M0–M2 and for the final paper.

In [14]:
# ============================================================
# 14. SAVE M3
# ============================================================

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "targets": TARGETS,
        "num_frames": NUM_FRAMES,
        "grid_size": GRID_SIZE,
        "patch_size": PATCH_SIZE,
        "patch_batch_size": PATCH_BATCH_SIZE,
        "seed": SEED
    },
    "/kaggle/working/M3_patch_cnn_average_pooling.pt"
)

with open(
    "/kaggle/working/M3_config.json",
    "w"
) as f:
    json.dump(
        {
            "model": "ResNet-50 + Patch Quality Head",
            "frames": NUM_FRAMES,
            "patch_grid": f"{GRID_SIZE}x{GRID_SIZE}",
            "patches_per_frame": NUM_PATCHES,
            "patch_size": PATCH_SIZE,
            "spatial_pooling": "average",
            "temporal_pooling": "average",
            "targets": TARGETS,
            "freeze_backbone": FREEZE_BACKBONE,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "seed": SEED
        },
        f,
        indent=2
    )

pd.DataFrame(history).to_csv(
    "/kaggle/working/M3_training_history.csv",
    index=False
)

print("M3 artifacts saved:")
for f in [
    "M3_patch_cnn_average_pooling.pt",
    "M3_config.json",
    "M3_training_history.csv",
    "M3_test_metrics.csv",
    "M3_test_predictions.csv",
    "M3_train_split.csv",
    "M3_test_split.csv"
]:
    print("/kaggle/working/" + f)


M3 artifacts saved:
/kaggle/working/M3_patch_cnn_average_pooling.pt
/kaggle/working/M3_config.json
/kaggle/working/M3_training_history.csv
/kaggle/working/M3_test_metrics.csv
/kaggle/working/M3_test_predictions.csv
/kaggle/working/M3_train_split.csv
/kaggle/working/M3_test_split.csv
